# Projeto Financeiro: Pipeline de Análise Financeira empresarial 

MVP: Engenharia de Dados

Autor: Tony Ribeiro Sá
Matrícula
Fonte de Dados: Sistema ERP - 90 Compor, de uso interno da ogranização 

# Objetivo

O projeto visa analisar valores e resultados financeiro da organização do segmento de construção pesada a partir do ano de 2025, considerando lançamentos futuros até 2039, classificados como "Provisão" financeira. 

* Qual o Total de Despesa no período do fluxo de caixa?
* Qual o Total de Receita no período do fluxo de caixa?
* Qual fornecedor a empresa gastou mais?
* Qual Cliente "Órgão Publico" pagou mais para a empresa?
* Resultado financeiro, a empresa está em crescimento, sim ou não? 




### IMPORTAÇÃO DE DADOS NECESSÁRIOS ###


In [0]:
df_receita = spark.read.csv(
  "/Volumes/mvp_pucrj/mpv_pucrj_trabalho/arquivos/Lista Analítica - Receita.csv",
  header=True,
  sep=";",
  encoding="UTF-8",
  multiLine=True
)


In [0]:
df_despesa = spark.read.csv(
  "/Volumes/mvp_pucrj/mpv_pucrj_trabalho/arquivos/Lista Analítica - Despesa.csv",
  header=True,
  sep=";",
  encoding="UTF-8",
  multiLine=True
)

df_despesa.printSchema()
df_despesa.show(5, truncate=False)


root
 |-- provisao_previsao: string (nullable = true)
 |-- agrupamento_de_locais: string (nullable = true)
 |-- empresa: string (nullable = true)
 |-- local_apropriacao: string (nullable = true)
 |-- cliente_fornecedor: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- dt_vencimento: string (nullable = true)
 |-- historico_apropriacao: string (nullable = true)
 |-- natureza: string (nullable = true)
 |-- vlr_apropriacao: string (nullable = true)

+-----------------+---------------------+--------------------------------+------------------------------+-----------------------------------------------------+-----+--------+-------------+------------------------------------------------------------------+--------------------------------------------------+---------------+
|provisao_previsao|agrupamento_de_locais|empresa                         |local_apropriacao             |cliente_fornecedor                                   |tipo |titulo 

# RECEITA


Foi realizada a padronização dos nomes das colunas dos dados de receita, removendo espaços, acentuação e caracteres especiais, além da conversão para letras minúsculas e uso de underscore. Essa etapa foi essencial para garantir compatibilidade com o ambiente Databricks e permitir a execução das consultas analíticas em SQL.

In [0]:
import re

def padronizar_colunas(df):
    for c in df.columns:
        novo = c.strip().lower()
        novo = novo.replace("/", "_")
        novo = re.sub(r"\s+", "_", novo)
        novo = re.sub(r"[áàãâ]", "a", novo)
        novo = re.sub(r"[éê]", "e", novo)
        novo = re.sub(r"[í]", "i", novo)
        novo = re.sub(r"[óôõ]", "o", novo)
        novo = re.sub(r"[ú]", "u", novo)
        novo = re.sub(r"[^\w]", "", novo)
        df = df.withColumnRenamed(c, novo)
    return df

df_receita = padronizar_colunas(df_receita)
print(df_receita.columns)


['tipo', 'agrupamento_de_locais', 'empresa', 'local_apropriacao', 'cliente_fornecedor', 'tipo_lancamento', 'titulo', 'dt_vencimento', 'historico_apropriacao', 'natureza', 'vlr_apropriacao']


Os valores monetários de receita foram normalizados por meio da remoção de símbolos, separadores e caracteres invisíveis, sendo posteriormente convertidos de forma segura para tipo numérico. Essa etapa foi fundamental para possibilitar a realização das análises financeiras no Databricks.

In [0]:
from pyspark.sql.functions import expr

df_receita = df_receita.withColumn(
    "valor_receita",
    expr("""
      try_cast(
        replace(
          replace(
            replace(
              replace(
                replace(vlr_apropriacao, '\u00A0', ''),
              ' ', ''),
            '.', ''),
          ',', '.'),
        'R$', '')
      as double)
    """)
)

df_receita.select("vlr_apropriacao", "valor_receita").show(10, truncate=False)


+---------------+-------------+
|vlr_apropriacao|valor_receita|
+---------------+-------------+
|  9018693,39   |9018693.39   |
|  900444,50    |900444.5     |
|  222913,34    |222913.34    |
|  430578,45    |430578.45    |
|  231452,23    |231452.23    |
|  52041,91     |52041.91     |
|  13274,74     |13274.74     |
|  2360489,65   |2360489.65   |
|  3000000,00   |3000000.0    |
|  216175,10    |216175.1     |
+---------------+-------------+
only showing top 10 rows


# CRIANDO VIEW DE RECEITA

In [0]:
df_receita.createOrReplaceTempView("receita")


# CONTANDO AS LINHAS DO RECEITA PARA CONFERENCIA DA VIEW

In [0]:
%sql
SELECT COUNT(*) AS linhas FROM receita;


linhas
120


# DESPESA

Foi realizada a padronização dos nomes das colunas dos dados despesa, removendo espaços, acentuação e caracteres especiais, além da conversão para letras minúsculas e uso de underscore. Essa etapa foi essencial para garantir compatibilidade com o ambiente Databricks e permitir a execução das consultas analíticas em SQL.

In [0]:
import re

def padronizar_colunas(df):
    for c in df.columns:
        novo = c.strip().lower()
        novo = novo.replace("/", "_")
        novo = re.sub(r"\s+", "_", novo)
        novo = re.sub(r"[áàãâ]", "a", novo)
        novo = re.sub(r"[éê]", "e", novo)
        novo = re.sub(r"[í]", "i", novo)
        novo = re.sub(r"[óôõ]", "o", novo)
        novo = re.sub(r"[ú]", "u", novo)
        novo = re.sub(r"[^\w]", "", novo)
        df = df.withColumnRenamed(c, novo)
    return df

df_despesa = padronizar_colunas(df_despesa)
print(df_despesa.columns)


['provisao_previsao', 'agrupamento_de_locais', 'empresa', 'local_apropriacao', 'cliente_fornecedor', 'tipo', 'titulo', 'dt_vencimento', 'historico_apropriacao', 'natureza', 'vlr_apropriacao']


Os valores monetários de receita foram normalizados por meio da remoção de símbolos, separadores e caracteres invisíveis, sendo posteriormente convertidos de forma segura para tipo numérico. Essa etapa foi fundamental para possibilitar a realização das análises financeiras no Databricks.

In [0]:
from pyspark.sql.functions import expr

df_despesa = df_despesa.withColumn(
    "valor_despesa",
    expr("""
      try_cast(
        replace(
          replace(
            replace(
              replace(
                replace(vlr_apropriacao, '\u00A0', ''),  -- NBSP invisível
              ' ', ''),                                 -- espaços normais
            '.', ''),                                   -- separador milhar
          ',', '.'),                                    -- separador decimal
        'R$', '')                                       -- símbolo monetário
      as double)
    """)
)

df_despesa.select("vlr_apropriacao", "valor_despesa").show(20, truncate=False)


+---------------+-------------+
|vlr_apropriacao|valor_despesa|
+---------------+-------------+
|4653,53        |4653.53      |
| 7086,86       |7086.86      |
| 3823,82       |3823.82      |
| 1990,00       |1990.0       |
| 1576,22       |1576.22      |
| 5576,16       |5576.16      |
| 25736,12      |25736.12     |
| 3882,87       |3882.87      |
| 3882,87       |3882.87      |
| 3882,87       |3882.87      |
| 20502,91      |20502.91     |
| 20788,70      |20788.7      |
| 21088,53      |21088.53     |
| 21393,00      |21393.0      |
| 21672,06      |21672.06     |
| 21985,58      |21985.58     |
| 22293,61      |22293.61     |
| 22616,77      |22616.77     |
| 22934,27      |22934.27     |
| 23267,36      |23267.36     |
+---------------+-------------+
only showing top 20 rows


# CRIANDO VIEW DE DESPESA

In [0]:
df_despesa.createOrReplaceTempView("despesa")


Totalizando as linhas de despesa para conferencia 

In [0]:
%sql
SELECT COUNT(*) AS linhas FROM despesa;


linhas
10717


# TOTALIZADORES PARA CONFERENCIA 

In [0]:
%sql
SHOW TABLES;


database,tableName,isTemporary
default,lista_analitica_despesa,false
default,lista_analitica_receita,false
default,total_despesa,false
default,total_receita,false
,despesa,true
,receita,true


Total de Linhas para Despesa

In [0]:
# garanta que os dataframes EXISTEM
df_receita.count()
df_despesa.count()


10717

Total de Linhas para Receita

In [0]:
%sql
SELECT COUNT(*) AS linhas_receita FROM receita;


linhas_receita
120


# 1 - Qual o Total de Despesa

In [0]:
%sql
SELECT
  CONCAT(
    'R$ ',
    FORMAT_NUMBER(total_despesa, 2)
  ) AS total_despesa
FROM total_despesa;


total_despesa
"R$ 91,561,789.47"


# 2 - Qual o Total de Receita

In [0]:
%sql
SELECT
  CONCAT(
    'R$ ',
    FORMAT_NUMBER(total_receita, 2)
  ) AS total_receita
FROM total_receita;



total_receita
"R$ 70,028,576.60"


# Qual fornecedor a empresa gastou mais em compra

In [0]:
%sql
SELECT
  cliente_fornecedor AS fornecedor,
  CONCAT(
    'R$ ',
    FORMAT_NUMBER(ROUND(SUM(valor_despesa), 2), 2)
  ) AS total_gasto
FROM despesa
WHERE valor_despesa IS NOT NULL
GROUP BY cliente_fornecedor
ORDER BY ROUND(SUM(valor_despesa), 2) DESC
LIMIT 1;



fornecedor,total_gasto
004476 - BRADESCO CONT. 237/3567/9790,"R$ 16,871,370.72"


#  LISTA DE FORNECEDOR TOP 10

In [0]:
%sql
WITH top10 AS (
  SELECT
    cliente_fornecedor AS fornecedor,
    ROUND(SUM(valor_despesa), 2) AS total_gasto,
    1 AS ordem
  FROM despesa
  WHERE valor_despesa IS NOT NULL
  GROUP BY cliente_fornecedor
  ORDER BY total_gasto DESC
  LIMIT 10
),
total_geral AS (
  SELECT
    'TOTAL GERAL' AS fornecedor,
    ROUND(SUM(valor_despesa), 2) AS total_gasto,
    2 AS ordem
  FROM despesa
  WHERE valor_despesa IS NOT NULL
)
SELECT
  fornecedor,
  CONCAT('R$ ', FORMAT_NUMBER(total_gasto, 2)) AS total_gasto
FROM (
  SELECT * FROM top10
  UNION ALL
  SELECT * FROM total_geral
)
ORDER BY ordem, total_gasto DESC;




fornecedor,total_gasto
000584 - BANCO SANTANDER,"R$ 7,712,322.58"
000686 - BRASIL ASFALTOS LTDA,"R$ 7,344,248.87"
001124 - CONSORCIO MAF / LIGA,"R$ 3,131,971.95"
000123 - BANCO ITAÚ,"R$ 2,704,435.66"
000575 - BANCO DO BRASIL,"R$ 2,572,281.02"
000949 - CIBER EQUIPAMENTOS RODOVIÁRIOS LTDA,"R$ 2,542,381.59"
000583 - BANCO SAFRA S/A,"R$ 2,374,365.46"
001637 - F W MAQUINAS DISTRIBUICAO E COMERCIO LTDA,"R$ 2,080,639.85"
004476 - BRADESCO CONT. 237/3567/9790,"R$ 16,871,370.72"
000578 - BANCO ITAU S.A.,"R$ 10,188,947.85"


# Qual Cliente "Órgão Publico" pagou mais para a empresa?

In [0]:
%sql
SELECT
  cliente_fornecedor AS cliente_orgao_publico,
  CONCAT(
    'R$ ',
    FORMAT_NUMBER(ROUND(SUM(valor_receita), 2), 2)
  ) AS total_pago
FROM receita
WHERE valor_receita IS NOT NULL
  AND (
    UPPER(cliente_fornecedor) LIKE '%ORG%'
    OR UPPER(cliente_fornecedor) LIKE '%PUBLIC%'
    OR UPPER(cliente_fornecedor) LIKE '%MUNIC%'
    OR UPPER(cliente_fornecedor) LIKE '%PREFEIT%'
    OR UPPER(cliente_fornecedor) LIKE '%ESTADO%'
    OR UPPER(cliente_fornecedor) LIKE '%GOVERNO%'
  )
GROUP BY cliente_fornecedor
ORDER BY ROUND(SUM(valor_receita), 2) DESC
LIMIT 1;


cliente_orgao_publico,total_pago
000002 - SUPERINTENDENCIA DE OBRAS PUBLICAS DE SALVADOR,"R$ 23,584,391.81"


# TOP 10 - CLIENTES MAIORES RECEITA

In [0]:
%sql
WITH top10_clientes AS (
  SELECT
    cliente_fornecedor AS cliente,
    ROUND(SUM(valor_receita), 2) AS total_receita,
    1 AS ordem
  FROM receita
  WHERE valor_receita IS NOT NULL
  GROUP BY cliente_fornecedor
  ORDER BY total_receita DESC
  LIMIT 10
),
total_geral AS (
  SELECT
    'TOTAL TOP 10' AS cliente,
    ROUND(SUM(total_receita), 2) AS total_receita,
    2 AS ordem
  FROM top10_clientes
)
SELECT
  cliente,
  CONCAT('R$ ', FORMAT_NUMBER(total_receita, 2)) AS total_receita
FROM (
  SELECT * FROM top10_clientes
  UNION ALL
  SELECT * FROM total_geral
)
ORDER BY ordem, total_receita DESC;


cliente,total_receita
000013 - MUNICIPIO DE PETROLINA,"R$ 9,958,950.19"
000020 - COMPANHIA DE DESENVOLVIMENTO DOS VALES DO SAO FRANCISCO E DO,"R$ 8,224,881.08"
000003 - MUNICIPIO DE VITORIA/ES,"R$ 6,464,628.30"
000016 - MUNICIPIO DE CAMACARI,"R$ 3,360,608.83"
000002 - SUPERINTENDENCIA DE OBRAS PUBLICAS DE SALVADOR,"R$ 23,584,391.81"
000018 - MUNICIPIO DE SAO FRANCISCO DO CONDE,"R$ 2,423,091.89"
000050 - GRAOS DO PIAUI CONCESSIONARIA DE RODOVIAS SPE S.A.,"R$ 2,200,000.00"
000010 - MUNICIPIO DE DIAS D'AVILA,"R$ 1,920,375.30"
000098 - MUNICIPIO DE CASA NOVA,"R$ 1,915,783.15"
000002 - MUNICIPIO DE JUAZEIRO,"R$ 1,894,726.00"


# Resultado financeiro, a empresa está em crescimento, sim ou não?

In [0]:
%sql
SELECT
  CONCAT('R$ ', FORMAT_NUMBER(r.total_receita, 2)) AS total_receita,
  CONCAT('R$ ', FORMAT_NUMBER(d.total_despesa, 2)) AS total_despesa,
  CONCAT(
    'R$ ',
    FORMAT_NUMBER(r.total_receita - d.total_despesa, 2)
  ) AS resultado_final,
  CASE
    WHEN r.total_receita - d.total_despesa >= 0 THEN 'POSITIVO'
    ELSE 'NEGATIVO'
  END AS status_resultado
FROM
  (SELECT SUM(valor_receita) AS total_receita FROM receita) r
CROSS JOIN
  (SELECT SUM(valor_despesa) AS total_despesa FROM despesa) d;


total_receita,total_despesa,resultado_final,status_resultado
"R$ 70,028,576.60","R$ 91,561,789.47","R$ -21,533,212.87",NEGATIVO


# PERÍODO DOS LANÇAMENTOS

In [0]:
%sql
SELECT
  MIN(TO_DATE(dt_vencimento, 'dd/MM/yyyy')) AS data_inicial,
  MAX(TO_DATE(dt_vencimento, 'dd/MM/yyyy')) AS data_final
FROM (
  SELECT dt_vencimento FROM receita
  UNION ALL
  SELECT dt_vencimento FROM despesa
);


data_inicial,data_final
2024-10-25,2033-10-23
